## Read data

demonstration data set from the UCR collection

In [2]:
import numpy as np

data = [np.load(f'../data/GestureMidAirD1/{variable}_{set_name}.npy')
        for variable in ['X', 'y'] for set_name in ['train', 'test']]

X_train, X_test, y_train, y_test = data

print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (208, 1, 360)
X_test dims:  (130, 1, 360)


## Define SelfEnsembling
The idea is to resize the input sequence to multiple lengths, 
pass them through the foundation model,
and use the concatenation of the outputs as the final embedding.

In [7]:
import torch
import torch.nn.functional as F


class OriginalPipeline(torch.nn.Module):
    def __init__(self, network, device='cuda'):
        super().__init__()
        self.network = network
        self.resize = lambda X: F.interpolate(X, size=512, mode='linear', align_corners=False)
    
    def forward(self, x, tokens=None):
        x = self.resize(x)
        return self.network(x)


class SelfEnsembling(torch.nn.Module):
    def __init__(self, network, sizes=[256, 512, 768, 1024], device='cuda'):
        super().__init__()
        self.network = network
        self.resize = lambda X: [F.interpolate(X, size=size, mode='linear', align_corners=False) for size in sizes]
        self.hidden_dim = self.network.hidden_dim * len(sizes)
    
    def forward(self, x, tokens=None):
        xs = self.resize(x)
        return torch.cat([self.network(x) for x in xs], axis=1)

## Evaluation Protocol

In [20]:
from mantis.architecture import MantisV1, MantisV2, Mantis8M
from mantis.trainer import MantisTrainer
from sklearn.ensemble import RandomForestClassifier
    
device = 'cpu' # set device

def evalute_model(network, name):
    # get the embeddings
    model = MantisTrainer(device=device, network=network) # init trainer
    Z_train = model.transform(X_train)
    Z_test = model.transform(X_test)

    # train the predictor
    predictor = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
    predictor.fit(Z_train, y_train)

    # evaluate the predictor
    y_pred = predictor.predict(Z_test)
    print(f'\t{name}: {np.mean(y_test == y_pred).round(4)}')

## Results

We are going to test the performance of all our 3 models with and without self-ensembling.
We show the performance comparison at each transformer layer (see `intermediate_layers.ipynb` for more details). We set `output_token='combined'`.

### Original Mantis

In [22]:
for i in range(6):
    network = MantisV1(device=device, return_transf_layer=i, output_token='combined') # init model
    network = network.from_pretrained("paris-noah/Mantis-8M") # load weights

    network_original = OriginalPipeline(network=network)
    # wrap the network with SelfEnsembling
    network_se = SelfEnsembling(network=network)

    print(f"Accuracy at layer = {i}")
    evalute_model(network_original, 'MantisV1')
    evalute_model(network_se, 'SE-MantisV1')

Accuracy at layer = 0
	MantisV1: 0.6769
	SE-MantisV1: 0.7
Accuracy at layer = 1
	MantisV1: 0.7
	SE-MantisV1: 0.7
Accuracy at layer = 2
	MantisV1: 0.7
	SE-MantisV1: 0.6923
Accuracy at layer = 3
	MantisV1: 0.7154
	SE-MantisV1: 0.7538
Accuracy at layer = 4
	MantisV1: 0.6846
	SE-MantisV1: 0.7
Accuracy at layer = 5
	MantisV1: 0.6923
	SE-MantisV1: 0.6846


### Mantis+

In [24]:
for i in range(6):
    network = MantisV1(device=device, return_transf_layer=i, output_token='combined') # init model
    network = network.from_pretrained("paris-noah/MantisPlus") # load weights

    network_original = OriginalPipeline(network=network)
    # wrap the network with SelfEnsembling
    network_se = SelfEnsembling(network=network)

    print(f"Accuracy at layer = {i}")
    evalute_model(network_original, 'Mantis+')
    evalute_model(network_se, 'SE-Mantis+')

Accuracy at layer = 0
	Mantis+: 0.6692
	SE-Mantis+: 0.7077
Accuracy at layer = 1
	Mantis+: 0.6923
	SE-Mantis+: 0.6846
Accuracy at layer = 2
	Mantis+: 0.6538
	SE-Mantis+: 0.6923
Accuracy at layer = 3
	Mantis+: 0.7
	SE-Mantis+: 0.6846
Accuracy at layer = 4
	Mantis+: 0.6615
	SE-Mantis+: 0.6692
Accuracy at layer = 5
	Mantis+: 0.7
	SE-Mantis+: 0.6923


### MantisV2

In [25]:
for i in range(6):
    network = MantisV2(device=device, return_transf_layer=i, output_token='combined') # init model
    network = network.from_pretrained("paris-noah/MantisV2") # load weights

    network_original = OriginalPipeline(network=network)
    # wrap the network with SelfEnsembling
    network_se = SelfEnsembling(network=network)

    print(f"Accuracy at layer = {i}")
    evalute_model(network_original, 'MantisV2')
    evalute_model(network_se, 'SE-MantisV2')

Accuracy at layer = 0
	MantisV2: 0.6385
	SE-MantisV2: 0.6769
Accuracy at layer = 1
	MantisV2: 0.7
	SE-MantisV2: 0.7
Accuracy at layer = 2
	MantisV2: 0.7308
	SE-MantisV2: 0.7231
Accuracy at layer = 3
	MantisV2: 0.7
	SE-MantisV2: 0.7077
Accuracy at layer = 4
	MantisV2: 0.7308
	SE-MantisV2: 0.7077
Accuracy at layer = 5
	MantisV2: 0.6615
	SE-MantisV2: 0.6846


## Take Home Message

In contrast to the intermediate representations, the benefits from self-ensembling are a bit more dataset-dependent. One can see that it may bring a nice improvement (e.g., 0.7538 vs 0.7154 for the layer = 3 of MantisV1), so it's a good hyperparameter to play with.